# Hénon Map — Jacobian and Eigenvalue Analysis

The **Hénon map** is a canonical 2D area-preserving (for suitable parameters) discrete map:
$$x_{n+1} = -y_n + 2(K - x_n^2), \quad y_{n+1} = x_n$$

It serves as a simple test case for pyna's Poincaré map analysis tools.

This notebook demonstrates:
1. Orbit integration of the Hénon map
2. Jacobian (tangent map) propagation along orbits
3. Eigenvalue analysis — distinguishing elliptic/hyperbolic behavior

**Source**: Adapted from `MCF_scripts/investigate_Jac_behaviour_by_Henon_map.ipynb` (Julia) and
`Jac_change_under_perturbation.ipynb` (Julia). Translated to Python for pyna.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eig

## 1. Hénon Map Definition

The map and its Jacobian matrix:
$$f(x,y) = \begin{pmatrix} -y + 2(K-x^2) \\ x \end{pmatrix}, \quad
Df(x,y) = \begin{pmatrix} -4x & -1 \\ 1 & 0 \end{pmatrix}$$

In [ ]:
def henon(xy, K):
    """One step of the Hénon map."""
    x, y = xy
    return np.array([-y + 2*(K - x**2), x])

def henon_jac(xy):
    """Jacobian of the Hénon map at point xy."""
    x, y = xy
    return np.array([[-4*x, -1.0],
                     [ 1.0,  0.0]])

def iterate_orbit(xy0, K, n_steps, outside_condition=None):
    """Iterate the Hénon map, returning the orbit array."""
    orbit = [xy0.copy()]
    xy = xy0.copy()
    for _ in range(n_steps):
        xy = henon(xy, K)
        if outside_condition is not None and outside_condition(xy):
            break
        orbit.append(xy.copy())
    return np.array(orbit)

def propagate_jacobian(orbit, K):
    """Propagate the tangent map along an orbit. Returns list of 2x2 Jacobian matrices."""
    J = np.eye(2)
    jacs = [J.copy()]
    for xy in orbit[:-1]:
        J = henon_jac(xy) @ J
        jacs.append(J.copy())
    return jacs

## 2. Phase Portrait — Multiple Orbits

In [ ]:
K = -0.17
n_steps = 5000

# Escape condition: leave a bounding box
def escape(xy):
    return abs(xy[0]) > 2 or abs(xy[1]) > 2

fig, ax = plt.subplots(figsize=(7, 7))

# Sample initial conditions
num_trajs = 20
x0s = np.linspace(-0.22, 0.04, num_trajs)
y0s = np.linspace(-0.22, -0.31, num_trajs)

for x0, y0 in zip(x0s, y0s):
    orbit = iterate_orbit(np.array([x0, y0]), K, n_steps, outside_condition=escape)
    ax.scatter(orbit[:, 0], orbit[:, 1], s=0.3, alpha=0.6)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Hénon Map Phase Portrait (K={K})')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Jacobian Eigenvalue Analysis

For a discrete map, the stability of a periodic orbit is determined by eigenvalues of
the accumulated Jacobian $J^{(n)} = Df^n(x_0)$:

- $|\lambda| = 1$ (complex): **elliptic** (stable island)
- $|\lambda| > 1$ or $< 1$ (real): **hyperbolic** (unstable saddle)

The Hénon map has $\det(Df) = 1$ everywhere, so eigenvalues come in reciprocal pairs.

In [ ]:
# Integrate one orbit and track eigenvalues of accumulated Jacobian
xy0 = np.array([-0.17, -0.28])
orbit = iterate_orbit(xy0, K, 500, outside_condition=escape)
jacs = propagate_jacobian(orbit, K)

eigenvalues = [np.linalg.eigvals(J) for J in jacs]
eig_real = [np.real(ev) for ev in eigenvalues]
eig_imag = [np.imag(ev) for ev in eigenvalues]
eig_mod  = [np.abs(ev) for ev in eigenvalues]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Eigenvalue modulus vs iteration
ax = axes[0]
steps = range(len(eigenvalues))
ax.semilogy(steps, [m[0] for m in eig_mod], label='|λ₁|')
ax.semilogy(steps, [m[1] for m in eig_mod], label='|λ₂|', linestyle='--')
ax.axhline(1.0, color='k', linestyle=':', alpha=0.5, label='|λ|=1')
ax.set_xlabel('Iteration')
ax.set_ylabel('|eigenvalue|')
ax.set_title('Eigenvalue Modulus of Accumulated Jacobian')
ax.legend()
ax.grid(True, alpha=0.3)

# Eigenvalues in complex plane
ax = axes[1]
theta = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, label='unit circle')
evs = np.array([eigenvalues[i] for i in range(0, len(eigenvalues), 10)])
ax.scatter(evs[:, 0].real, evs[:, 0].imag, s=5, alpha=0.5, label='λ₁')
ax.scatter(evs[:, 1].real, evs[:, 1].imag, s=5, alpha=0.5, label='λ₂')
ax.set_xlabel('Re(λ)')
ax.set_ylabel('Im(λ)')
ax.set_title('Eigenvalues in Complex Plane')
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Determinant check (should be 1 for area-preserving map)
dets = [np.linalg.det(J) for J in jacs]
print(f"det(J) range: [{min(np.abs(dets)):.6f}, {max(np.abs(dets)):.6f}] — should be 1 for each step")

## 4. Jacobian Under Parameter Perturbation

How do Jacobian eigenvalues change when the map parameter $K$ is slightly perturbed?
This is relevant to understanding how stability changes under perturbations of the magnetic field.

In [ ]:
K_values = np.linspace(-0.25, -0.10, 30)
max_eig_mods = []

for K_val in K_values:
    orbit = iterate_orbit(xy0, K_val, 200, outside_condition=escape)
    if len(orbit) < 10:
        max_eig_mods.append(np.nan)
        continue
    jacs = propagate_jacobian(orbit, K_val)
    final_eigs = np.abs(np.linalg.eigvals(jacs[-1]))
    max_eig_mods.append(np.max(final_eigs))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(K_values, max_eig_mods, 'o-')
ax.axhline(1.0, color='k', linestyle='--', alpha=0.5, label='|λ|=1 (marginal stability)')
ax.set_xlabel('K')
ax.set_ylabel('max|λ| after 200 steps')
ax.set_title('Max Eigenvalue Modulus vs K')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()